# 05 — Ablation Study & Baseline Karşılaştırma

**Amaç:** ESWA makalesi için modelin her bileşeninin katkısını kanıtlamak.

## Deneyler

### A. Baselines (GNN olmadan)
1. **Logistic Regression** — ülke özelliklerinin ortalaması ile
2. **Random Forest** — aynı özellikler
3. **MLP** — aynı özellikler (GNN'siz neural baseline)

### B. Ablation (GNN bileşenleri)
4. **No Pretrain** — random init encoder
5. **No Bilinear** — sadece DeepSets (pairwise etkileşim yok)
6. **No Multi-task** — sadece BCE (duration/cohesion yardımcı loss yok)
7. **Single Edge** — sadece alliance kenarları (4 yerine 1 ilişki tipi)

### C. Full Model (referans)
8. **CoalitionGNN** — tam model (04_model sonuçları)

**Metrikler:** ROC-AUC, PR-AUC, F1 (optimal eşik) — hepsi test seti (2010-2016) üzerinde

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
SNAP_DIR = os.path.join(PROJECT_DIR, 'data/snapshots')
CANON_DIR = os.path.join(PROJECT_DIR, 'data/canonical')
COAL_DIR = os.path.join(PROJECT_DIR, 'data/coalitions')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')

START_YEAR = 1946
TRAIN_END = 1999
VAL_END = 2009
END_YEAR = 2016

In [ ]:
!pip install -q torch torch_geometric numpy pandas tqdm scikit-learn matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import random
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Veri yükleme (04_model ile aynı)
EDGE_TYPES = ['allies', 'trades', 'votes_with', 'conflicts_with']
NUM_RELATIONS = len(EDGE_TYPES)

def load_snapshot(year):
    path = os.path.join(SNAP_DIR, f'graph_{year}.pt')
    if not os.path.exists(path):
        return None
    data = torch.load(path, weights_only=False)
    x = data['country'].x
    cow_codes = data['country'].cow_code.numpy()
    edge_index_list, edge_type_list = [], []
    for rel_idx, rel_name in enumerate(EDGE_TYPES):
        key = ('country', rel_name, 'country')
        if key in data.edge_types:
            ei = data[key].edge_index
            edge_index_list.append(ei)
            edge_type_list.append(torch.full((ei.shape[1],), rel_idx, dtype=torch.long))
    if not edge_index_list:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_type = torch.zeros(0, dtype=torch.long)
    else:
        edge_index = torch.cat(edge_index_list, dim=1)
        edge_type = torch.cat(edge_type_list)
    return {'x': x, 'edge_index': edge_index, 'edge_type': edge_type,
            'cow_codes': cow_codes, 'code_to_idx': {int(c): i for i, c in enumerate(cow_codes)}, 'year': year}

snapshots = {}
for year in range(START_YEAR, END_YEAR + 1):
    snap = load_snapshot(year)
    if snap is not None:
        snapshots[year] = snap

NUM_FEATURES = next(iter(snapshots.values()))['x'].shape[1]
print(f'Snapshot: {len(snapshots)}, Özellik: {NUM_FEATURES}')

# Koalisyon örnekleri
pos_df = pd.read_parquet(os.path.join(COAL_DIR, 'positive_samples.parquet'))
neg_df = pd.read_parquet(os.path.join(COAL_DIR, 'negative_samples.parquet'))

def parse_members(members_str):
    if isinstance(members_str, list): return [int(x) for x in members_str]
    if isinstance(members_str, str): return [int(x) for x in members_str.split(',')]
    return []

def build_samples(df, label):
    samples = []
    for _, row in df.iterrows():
        year = int(row['year'])
        if year not in snapshots: continue
        members = parse_members(row['members_str'])
        code_to_idx = snapshots[year]['code_to_idx']
        valid_members = [m for m in members if m in code_to_idx]
        if len(valid_members) < 2: continue
        samples.append({
            'year': year, 'members': valid_members,
            'member_idx': [code_to_idx[m] for m in valid_members],
            'label': label,
            'duration': float(row.get('total_duration', 0)) if label == 1 else 0.0,
            'cohesion': float(row.get('internal_vote_agreement', 0) or 0) if label == 1 else 0.0,
        })
    return samples

all_samples = build_samples(pos_df, 1) + build_samples(neg_df, 0)
train_samples = [s for s in all_samples if s['year'] <= TRAIN_END]
val_samples = [s for s in all_samples if TRAIN_END < s['year'] <= VAL_END]
test_samples = [s for s in all_samples if s['year'] > VAL_END]

print(f'Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}')

## A. Baselines (GNN Olmadan)

Koalisyon üyelerinin ham özelliklerinin **ortalaması** ile özellik vektörü oluşturulur.  
GNN yok → graf yapısı kullanılmıyor → sadece ülke özellikleri.

In [ ]:
# Baseline özellik çıkarımı: üye özelliklerinin mean + std + max + min + size
def extract_baseline_features(sample):
    """Koalisyon üyelerinin ham özelliklerinden feature vektörü oluştur."""
    year = sample['year']
    snap = snapshots[year]
    x = snap['x'].numpy()  # (N, D)
    member_idx = sample['member_idx']
    
    member_features = x[member_idx]  # (K, D)
    
    # İstatistiksel özetler
    feat_mean = member_features.mean(axis=0)  # (D,)
    feat_std = member_features.std(axis=0)    # (D,)
    feat_max = member_features.max(axis=0)    # (D,)
    feat_min = member_features.min(axis=0)    # (D,)
    
    # Koalisyon boyutu (normalize)
    size_feat = np.array([len(member_idx) / 20.0])  # max ~20 üye
    
    return np.concatenate([feat_mean, feat_std, feat_max, feat_min, size_feat])


print('Baseline özellik çıkarımı...')
X_train = np.array([extract_baseline_features(s) for s in tqdm(train_samples)])
y_train = np.array([s['label'] for s in train_samples])

X_val = np.array([extract_baseline_features(s) for s in val_samples])
y_val = np.array([s['label'] for s in val_samples])

X_test = np.array([extract_baseline_features(s) for s in test_samples])
y_test = np.array([s['label'] for s in test_samples])

# NaN temizle
X_train = np.nan_to_num(X_train, 0)
X_val = np.nan_to_num(X_val, 0)
X_test = np.nan_to_num(X_test, 0)

# Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f'Özellik boyutu: {X_train.shape[1]} (4×{NUM_FEATURES} + 1)')
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

In [ ]:
def evaluate_model(y_true, y_probs, name):
    """Metrik hesapla ve sonuçları döndür."""
    auc = roc_auc_score(y_true, y_probs)
    pr_auc = average_precision_score(y_true, y_probs)
    
    # Optimal F1 eşiği
    prec_vals, rec_vals, thresholds = precision_recall_curve(y_true, y_probs)
    f1_vals = 2 * prec_vals[:-1] * rec_vals[:-1] / (prec_vals[:-1] + rec_vals[:-1] + 1e-8)
    best_f1 = f1_vals.max()
    
    return {'name': name, 'AUC': auc, 'PR-AUC': pr_auc, 'F1': best_f1}


results = []

# 1. Logistic Regression
print('Training: Logistic Regression...')
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED, class_weight='balanced')
lr_model.fit(X_train, y_train)
lr_probs = lr_model.predict_proba(X_test)[:, 1]
results.append(evaluate_model(y_test, lr_probs, 'Logistic Regression'))
print(f'  AUC={results[-1]["AUC"]:.4f}, PR-AUC={results[-1]["PR-AUC"]:.4f}, F1={results[-1]["F1"]:.4f}')

# 2. Random Forest
print('Training: Random Forest...')
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=5,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_probs = rf_model.predict_proba(X_test)[:, 1]
results.append(evaluate_model(y_test, rf_probs, 'Random Forest'))
print(f'  AUC={results[-1]["AUC"]:.4f}, PR-AUC={results[-1]["PR-AUC"]:.4f}, F1={results[-1]["F1"]:.4f}')

# 3. MLP
print('Training: MLP...')
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64), max_iter=500,
    early_stopping=True, validation_fraction=0.15,
    random_state=SEED, alpha=1e-4
)
mlp_model.fit(X_train, y_train)
mlp_probs = mlp_model.predict_proba(X_test)[:, 1]
results.append(evaluate_model(y_test, mlp_probs, 'MLP'))
print(f'  AUC={results[-1]["AUC"]:.4f}, PR-AUC={results[-1]["PR-AUC"]:.4f}, F1={results[-1]["F1"]:.4f}')

print('\n✅ Baselines tamamlandı')

## B. Ablation Study

Her ablation'da tek bir bileşen çıkarılıp model baştan eğitilir.

In [ ]:
# Model tanımları (ablation varyantları)

class RGCNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, dropout=0.2):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, out_channels, num_relations)
        self.dropout = dropout
        self.norm1 = nn.LayerNorm(hidden_channels)
        self.norm2 = nn.LayerNorm(out_channels)
    
    def forward(self, x, edge_index, edge_type):
        h = self.conv1(x, edge_index, edge_type)
        h = self.norm1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, edge_index, edge_type)
        h = self.norm2(h)
        return h


class CoalitionHeadFull(nn.Module):
    """Tam coalition head (DeepSets + bilinear + aux)."""
    def __init__(self, embed_dim, hidden_dim=64, use_bilinear=True):
        super().__init__()
        self.use_bilinear = use_bilinear
        self.set_mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1),
        )
        if use_bilinear:
            self.bilinear = nn.Bilinear(embed_dim, embed_dim, 1)
            self.lambda_pair = 0.5
        self.duration_head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1), nn.Softplus())
        self.cohesion_head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1), nn.Sigmoid())
    
    def forward(self, z_members):
        K = z_members.shape[0]
        z_pool = z_members.mean(dim=0, keepdim=True)
        set_score = self.set_mlp(z_pool).squeeze()
        
        if self.use_bilinear and K >= 2:
            idx_i, idx_j = torch.triu_indices(K, K, offset=1, device=z_members.device)
            pair_scores = self.bilinear(z_members[idx_i], z_members[idx_j]).squeeze(-1)
            score = set_score + self.lambda_pair * pair_scores.mean()
        else:
            score = set_score
        
        duration_pred = self.duration_head(z_pool).squeeze()
        cohesion_pred = self.cohesion_head(z_pool).squeeze()
        return score, duration_pred, cohesion_pred


class AblationModel(nn.Module):
    def __init__(self, num_features, hidden_dim, embed_dim, num_relations,
                 use_bilinear=True, dropout=0.2):
        super().__init__()
        self.encoder = RGCNEncoder(num_features, hidden_dim, embed_dim, num_relations, dropout)
        self.head = CoalitionHeadFull(embed_dim, hidden_dim=64, use_bilinear=use_bilinear)
    
    def forward_score(self, snap, member_idx, device):
        x = snap['x'].to(device)
        ei = snap['edge_index'].to(device)
        et = snap['edge_type'].to(device)
        z = self.encoder(x, ei, et)
        idx = torch.tensor(member_idx, dtype=torch.long, device=device)
        return self.head(z[idx])


print('✅ Ablation model sınıfları tanımlandı')

In [ ]:
# Hızlı eğitim fonksiyonu (ablation için)
POS_WEIGHT = 2.5
BATCH_SIZE = 64

def train_ablation(model, train_samples, val_samples, snapshots, device,
                   use_multitask=True, total_epochs=80, patience=20,
                   lr=5e-4, pretrain_path=None):
    """Ablation modeli eğit ve en iyi val F1'deki modeli döndür."""
    
    # Pretrain yükle (varsa)
    if pretrain_path and os.path.exists(pretrain_path):
        ckpt = torch.load(pretrain_path, weights_only=False)
        model.encoder.load_state_dict(ckpt['encoder_state_dict'])
    
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    
    best_val_f1 = 0.0
    best_state = None
    patience_counter = 0
    pos_weight = torch.tensor([POS_WEIGHT], device=device)
    
    for epoch in range(1, total_epochs + 1):
        # Train
        model.train()
        indices = list(range(len(train_samples)))
        random.shuffle(indices)
        
        for i in range(0, len(indices), BATCH_SIZE):
            batch = [train_samples[j] for j in indices[i:i+BATCH_SIZE]]
            optimizer.zero_grad()
            
            scores, labels = [], []
            dur_preds, dur_trues = [], []
            coh_preds, coh_trues = [], []
            
            for s in batch:
                score, dur, coh = model.forward_score(snapshots[s['year']], s['member_idx'], device)
                scores.append(score)
                labels.append(s['label'])
                if s['label'] == 1 and use_multitask:
                    dur_preds.append(dur)
                    dur_trues.append(s['duration'])
                    coh_preds.append(coh)
                    coh_trues.append(s['cohesion'])
            
            scores_t = torch.stack(scores)
            labels_t = torch.tensor(labels, dtype=torch.float, device=device)
            loss = F.binary_cross_entropy_with_logits(scores_t, labels_t, pos_weight=pos_weight)
            
            if use_multitask and dur_preds:
                dur_loss = F.mse_loss(torch.stack(dur_preds),
                                      torch.log1p(torch.tensor(dur_trues, device=device)))
                loss = loss + 0.3 * dur_loss
                if coh_trues:
                    coh_t = torch.tensor(coh_trues, device=device)
                    valid = coh_t > 0
                    if valid.sum() > 0:
                        coh_loss = F.mse_loss(torch.stack(coh_preds)[valid], coh_t[valid])
                        loss = loss + 0.3 * coh_loss
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        # Val eval (her 5 epoch'ta)
        if epoch % 5 == 0 or epoch == 1:
            model.eval()
            val_scores, val_labels = [], []
            with torch.no_grad():
                for s in val_samples:
                    score, _, _ = model.forward_score(snapshots[s['year']], s['member_idx'], device)
                    val_scores.append(score.item())
                    val_labels.append(s['label'])
            
            val_probs = 1 / (1 + np.exp(-np.array(val_scores)))
            val_labels_arr = np.array(val_labels)
            prec_v, rec_v, thr_v = precision_recall_curve(val_labels_arr, val_probs)
            f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
            val_f1 = f1_v.max() if len(f1_v) > 0 else 0.0
            
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 5
            
            if patience_counter >= patience:
                break
    
    # En iyi modeli geri yükle
    if best_state:
        model.load_state_dict(best_state)
    model.to(device)
    
    return model, best_val_f1


def evaluate_gnn_model(model, test_samples, snapshots, device):
    """GNN modeli test setinde değerlendir."""
    model.eval()
    scores, labels = [], []
    with torch.no_grad():
        for s in test_samples:
            score, _, _ = model.forward_score(snapshots[s['year']], s['member_idx'], device)
            scores.append(score.item())
            labels.append(s['label'])
    
    probs = 1 / (1 + np.exp(-np.array(scores)))
    labels = np.array(labels)
    
    auc = roc_auc_score(labels, probs)
    pr_auc = average_precision_score(labels, probs)
    prec_v, rec_v, _ = precision_recall_curve(labels, probs)
    f1_vals = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
    best_f1 = f1_vals.max()
    
    return {'AUC': auc, 'PR-AUC': pr_auc, 'F1': best_f1}


print('✅ Eğitim/değerlendirme fonksiyonları hazır')

In [ ]:
# === ABLATION 1: No Pretrain (random initialization) ===
print('='*60)
print('ABLATION 1: No Pretrain')
print('='*60)

model_no_pretrain = AblationModel(
    NUM_FEATURES, 128, 128, NUM_RELATIONS, use_bilinear=True
)
model_no_pretrain, val_f1 = train_ablation(
    model_no_pretrain, train_samples, val_samples, snapshots, device,
    use_multitask=True, pretrain_path=None  # pretrain YOK
)
res = evaluate_gnn_model(model_no_pretrain, test_samples, snapshots, device)
res['name'] = 'No Pretrain'
results.append(res)
print(f'  Val F1={val_f1:.4f} | Test: AUC={res["AUC"]:.4f}, PR-AUC={res["PR-AUC"]:.4f}, F1={res["F1"]:.4f}')

In [ ]:
# === ABLATION 2: No Bilinear (sadece DeepSets) ===
print('='*60)
print('ABLATION 2: No Bilinear (DeepSets only)')
print('='*60)

model_no_bilinear = AblationModel(
    NUM_FEATURES, 128, 128, NUM_RELATIONS, use_bilinear=False  # bilinear YOK
)
model_no_bilinear, val_f1 = train_ablation(
    model_no_bilinear, train_samples, val_samples, snapshots, device,
    use_multitask=True, pretrain_path=os.path.join(MODEL_DIR, 'encoder_pretrained.pt')
)
res = evaluate_gnn_model(model_no_bilinear, test_samples, snapshots, device)
res['name'] = 'No Bilinear'
results.append(res)
print(f'  Val F1={val_f1:.4f} | Test: AUC={res["AUC"]:.4f}, PR-AUC={res["PR-AUC"]:.4f}, F1={res["F1"]:.4f}')

In [ ]:
# === ABLATION 3: No Multi-task (sadece BCE) ===
print('='*60)
print('ABLATION 3: No Multi-task (BCE only)')
print('='*60)

model_no_mt = AblationModel(
    NUM_FEATURES, 128, 128, NUM_RELATIONS, use_bilinear=True
)
model_no_mt, val_f1 = train_ablation(
    model_no_mt, train_samples, val_samples, snapshots, device,
    use_multitask=False,  # multi-task YOK
    pretrain_path=os.path.join(MODEL_DIR, 'encoder_pretrained.pt')
)
res = evaluate_gnn_model(model_no_mt, test_samples, snapshots, device)
res['name'] = 'No Multi-task'
results.append(res)
print(f'  Val F1={val_f1:.4f} | Test: AUC={res["AUC"]:.4f}, PR-AUC={res["PR-AUC"]:.4f}, F1={res["F1"]:.4f}')

In [ ]:
# === ABLATION 4: Single Edge Type (sadece alliance) ===
print('='*60)
print('ABLATION 4: Single Edge Type (allies only)')
print('='*60)

# Snapshot'ları sadece alliance kenarlarıyla yeniden oluştur
snapshots_single = {}
for year, snap in snapshots.items():
    data = torch.load(os.path.join(SNAP_DIR, f'graph_{year}.pt'), weights_only=False)
    key = ('country', 'allies', 'country')
    if key in data.edge_types:
        ei = data[key].edge_index
        et = torch.zeros(ei.shape[1], dtype=torch.long)  # tek tip = 0
    else:
        ei = torch.zeros((2, 0), dtype=torch.long)
        et = torch.zeros(0, dtype=torch.long)
    
    snapshots_single[year] = {
        'x': snap['x'], 'edge_index': ei, 'edge_type': et,
        'cow_codes': snap['cow_codes'], 'code_to_idx': snap['code_to_idx'], 'year': year
    }

model_single_edge = AblationModel(
    NUM_FEATURES, 128, 128, num_relations=1, use_bilinear=True  # 1 ilişki tipi
)
model_single_edge, val_f1 = train_ablation(
    model_single_edge, train_samples, val_samples, snapshots_single, device,
    use_multitask=True, pretrain_path=None  # farklı num_relations, pretrain uyumsuz
)
res = evaluate_gnn_model(model_single_edge, test_samples, snapshots_single, device)
res['name'] = 'Single Edge (allies)'
results.append(res)
print(f'  Val F1={val_f1:.4f} | Test: AUC={res["AUC"]:.4f}, PR-AUC={res["PR-AUC"]:.4f}, F1={res["F1"]:.4f}')

## C. Sonuç Tablosu

In [ ]:
# Full model sonuçlarını ekle (04_model'den)
results.append({
    'name': 'CoalitionGNN (Full)',
    'AUC': 0.8165,
    'PR-AUC': 0.6961,
    'F1': 0.6283,
})

# Sonuç tablosu
print('\n' + '='*70)
print(f'{"ABLATION STUDY — SONUÇ TABLOSU":^70}')
print(f'{"Test Seti (2010-2016), 358 örnek":^70}')
print('='*70)
print(f'\n  {"Model":<25} {"ROC-AUC":<10} {"PR-AUC":<10} {"F1(opt)":<10}')
print(f'  {"-"*55}')

# Sırala: full model en üstte, sonra ablations, sonra baselines
# Grupla
baselines = [r for r in results if r['name'] in ['Logistic Regression', 'Random Forest', 'MLP']]
ablations = [r for r in results if r['name'] in ['No Pretrain', 'No Bilinear', 'No Multi-task', 'Single Edge (allies)']]
full = [r for r in results if r['name'] == 'CoalitionGNN (Full)']

print(f'  {"— FULL MODEL —":<25}')
for r in full:
    print(f'  {r["name"]:<25} {r["AUC"]:<10.4f} {r["PR-AUC"]:<10.4f} {r["F1"]:<10.4f}')

print(f'\n  {"— ABLATIONS —":<25}')
for r in sorted(ablations, key=lambda x: x['AUC'], reverse=True):
    diff = r['AUC'] - 0.8165
    arrow = f'({diff:+.3f})'
    print(f'  {r["name"]:<25} {r["AUC"]:<10.4f} {r["PR-AUC"]:<10.4f} {r["F1"]:<10.4f} {arrow}')

print(f'\n  {"— BASELINES (no GNN) —":<25}')
for r in sorted(baselines, key=lambda x: x['AUC'], reverse=True):
    diff = r['AUC'] - 0.8165
    arrow = f'({diff:+.3f})'
    print(f'  {r["name"]:<25} {r["AUC"]:<10.4f} {r["PR-AUC"]:<10.4f} {r["F1"]:<10.4f} {arrow}')

print(f'\n  {"-"*55}')
print(f'  Full model AUC farkı: baselines ortalamasından +{0.8165 - np.mean([r["AUC"] for r in baselines]):.3f}')
print(f'\n{"="*70}')
print('YORUM:')
print('  - GNN vs baselines: graf yapısının katkısını gösterir')
print('  - No Pretrain: self-supervised pretraining\'in değerini gösterir')
print('  - No Bilinear: pairwise etkileşimin (üye çiftleri) katkısını gösterir')
print('  - No Multi-task: yardımcı görevlerin regularization etkisini gösterir')
print('  - Single Edge: heterojen kenar tiplerinin (4 vs 1) katkısını gösterir')

In [ ]:
# Görselleştirme
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

all_results = full + sorted(ablations, key=lambda x: x['AUC'], reverse=True) + sorted(baselines, key=lambda x: x['AUC'], reverse=True)
names = [r['name'] for r in all_results]
aucs = [r['AUC'] for r in all_results]
f1s = [r['F1'] for r in all_results]

x_pos = np.arange(len(names))
width = 0.35

bars1 = ax.bar(x_pos - width/2, aucs, width, label='ROC-AUC', color='#3498db', alpha=0.8)
bars2 = ax.bar(x_pos + width/2, f1s, width, label='F1 (optimal)', color='#e74c3c', alpha=0.8)

ax.set_ylabel('Score')
ax.set_title('Ablation Study — Model Bileşenlerinin Katkısı')
ax.set_xticks(x_pos)
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=9)
ax.legend()
ax.set_ylim(0.4, 0.95)
ax.grid(True, alpha=0.3, axis='y')

# Full model çizgisi
ax.axhline(y=0.8165, color='blue', linestyle='--', alpha=0.4, label='Full AUC')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'ablation_results.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Ablation study tamamlandı. Sonuçlar kaydedildi.')